# 02b — FRED Resample Sensitivity: `.last()` vs `.mean()`

**Purpose.** Standalone diagnostic, not part of the ingest pipeline. Both
`02_fred_ingest_clean.ipynb` (`.mean()`) and `03_teammate_pandas_comparison.ipynb`
(`.last()`, matching the teammate) now key `MORTGAGE30US` to **month-end**, so the
two pipelines differ only in aggregation. This notebook quantifies how much that
one difference actually moves `mortgage_rate`, so the reconciliation flagged in
notebook 03's cell 12 has real numbers behind it before Phase 4 feature
engineering picks one.

Reads the same raw weekly file both ingest notebooks use
(`../data/raw/fred/mortgage30us.csv`) and writes nothing back to Postgres — read-only
comparison, no `mortgage_monthly` writes.


## 1. Setup

In [ ]:
import os
import pandas as pd

pd.set_option("display.max_columns", 20)


## 2. Load weekly rates

Same raw file, same parsing as `02_fred_ingest_clean.ipynb` / `03_teammate_pandas_comparison.ipynb`.


In [ ]:
RAW_PATH = "../data/raw/fred/mortgage30us.csv"
assert os.path.exists(RAW_PATH), f"Expected raw FRED file at {RAW_PATH} -- run 02_fred_ingest_clean.ipynb's download cell first."

weekly = pd.read_csv(RAW_PATH)
weekly.columns = ["date", "mortgage_rate"]
weekly["date"] = pd.to_datetime(weekly["date"])
weekly["mortgage_rate"] = pd.to_numeric(weekly["mortgage_rate"], errors="coerce")
weekly = weekly.dropna(subset=["mortgage_rate"]).sort_values("date").reset_index(drop=True)

print("Weekly rows:", len(weekly))
print("Date range:", weekly["date"].min().date(), "to", weekly["date"].max().date())
weekly.head()


## 3. Resample both ways, same month-end key

Both use `.resample("ME")` -- only the aggregation function changes. This isolates
the one variable notebook 03's cell 12 says the two pipelines actually differ on.


In [ ]:
s = weekly.set_index("date")["mortgage_rate"]

monthly_last = s.resample("ME").last().rename("rate_last").to_frame()
monthly_mean = s.resample("ME").mean().rename("rate_mean").to_frame()

compare = monthly_last.join(monthly_mean, how="outer")
compare["diff"] = compare["rate_last"] - compare["rate_mean"]
compare["abs_diff"] = compare["diff"].abs()

compare.index.name = "month"
compare = compare.reset_index()
compare.head()


## 4. How much does it actually move `mortgage_rate`?

Summary stats on `abs_diff`, plus the months where the two conventions disagree
most -- these are the months most likely to matter if the rate-of-change feature
(Phase 4) is sensitive to which aggregation was used.


In [ ]:
n_months = len(compare)
print(f"Months compared: {n_months:,}")
print(f"Mean |diff|:   {compare['abs_diff'].mean():.4f} pp")
print(f"Median |diff|: {compare['abs_diff'].median():.4f} pp")
print(f"Max |diff|:    {compare['abs_diff'].max():.4f} pp")
print(f"Corr(rate_last, rate_mean): {compare['rate_last'].corr(compare['rate_mean']):.6f}")

# Months with the biggest disagreement -- typically fast-moving-rate months,
# where "last observed" and "average over the month" diverge most.
compare.reindex(compare["abs_diff"].sort_values(ascending=False).index).head(10)


## 5. Rate-of-change sensitivity

The feature that actually feeds the model (per the Team Reference doc's field
scope) is `MORTGAGE30US` rate-of-change, not the level itself -- so the more
relevant question is whether the two conventions still agree on month-over-month
*direction* and magnitude of change, not just on the level.


In [ ]:
compare["chg_last"] = compare["rate_last"].diff()
compare["chg_mean"] = compare["rate_mean"].diff()
compare["chg_diff"] = compare["chg_last"] - compare["chg_mean"]

sign_mismatch = (
    compare.dropna(subset=["chg_last", "chg_mean"])
    .loc[lambda d: (d["chg_last"] > 0) != (d["chg_mean"] > 0)]
)

print(f"Months where month-over-month direction disagrees: {len(sign_mismatch)} / {compare['chg_last'].notna().sum()}")
print(f"Mean |chg_diff|: {compare['chg_diff'].abs().mean():.4f} pp")
sign_mismatch[["month", "chg_last", "chg_mean"]].head(10)


## 6. Takeaway

Fill in after running against the current data: whether `.last()` vs `.mean()`
is close enough to not matter for Phase 4, or whether notebook 03's flagged
reconciliation needs to become a real decision (and if so, which convention wins
and why) before feature engineering starts.
